# 25 GSPO 为什么使用序列级 ratio，如何手写它？

## 面试回答主线

GSPO 的直觉是把一条完整回答作为策略更新与裁剪的基本单位：先汇总该回答的 log-prob 变化得到序列级 ratio，再结合组内相对优势做 clip surrogate。这样避免把同一回答拆成许多 token 后，长序列因 token 数更多而获得不同的隐式权重。实现时必须明确采用 sum 还是长度归一化 log-prob，并记录 ratio 分布、clip fraction 和组奖励。实验对两道客服处理 prompt 各采样三条完整回复，比较未归一化序列 ratio 与按长度平均的序列 ratio。

**核心公式：** 一种序列级写法为 $r_i=\exp((\log\pi_\theta(y_i|x)-\log\pi_{old}(y_i|x))/|y_i|)$，$L=\min(r_iA_i,\operatorname{clip}(r_i,1\pm\epsilon)A_i)$。具体归一化约定必须固定。

后续依次展示同数据基线、手写核心状态/概率、结果表、真实失败与修复。数值仅用于机制验证。


## 真实案例

数据是六条脱敏客服 prompt，每条含 chosen/rejected 回答；注意力主题会将它们映射成流式键值事件。字段语义和失败模式与真实系统一致，但样本规模不能代表线上效果。


In [1]:
import math  # 导入数学函数实现概率和复杂度公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的弃用提示。
import torch  # 导入张量和自动微分基础能力。
import torch.nn as nn  # 导入模块基类以显式定义网络。
torch.manual_seed(41)  # 固定随机种子保证输出可复现。
torch.set_num_threads(1)  # 固定小实验 CPU 线程数。
samples = [  # 定义六条可读的 prompt、候选回复或流式事件。
    {'id': 'P01', 'prompt': '支付重复扣款怎么处理？', 'chosen': '核验订单后原路退款。', 'rejected': '无需核验直接忽略。'},  # 退款决策样本。
    {'id': 'P02', 'prompt': '发现陌生转账怎么办？', 'chosen': '立即冻结并核验身份。', 'rejected': '等待下个账单周期。'},  # 账户安全样本。
    {'id': 'P03', 'prompt': '收不到登录验证码？', 'chosen': '检查手机号并重发。', 'rejected': '建议注销账户。'},  # 登录支持样本。
    {'id': 'P04', 'prompt': '地址如何修改？', 'chosen': '在发货前更新地址。', 'rejected': '永久不可修改。'},  # 售后样本。
    {'id': 'P05', 'prompt': '银行卡被盗刷？', 'chosen': '冻结卡并保留证据。', 'rejected': '继续正常使用。'},  # 风险样本。
    {'id': 'P06', 'prompt': '发票抬头写错？', 'chosen': '按规则更正抬头。', 'rejected': '删除全部订单。'},  # 账单样本。
]  # 结束可读数据定义。
print('教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。')  # 声明实验边界。
for row in samples:  # 逐条展示 prompt/chosen/rejected。
    print(f"{row['id']} | 问题={row['prompt']} | chosen={row['chosen']} | rejected={row['rejected']}")  # 输出真实语义样本。


教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。
P01 | 问题=支付重复扣款怎么处理？ | chosen=核验订单后原路退款。 | rejected=无需核验直接忽略。
P02 | 问题=发现陌生转账怎么办？ | chosen=立即冻结并核验身份。 | rejected=等待下个账单周期。
P03 | 问题=收不到登录验证码？ | chosen=检查手机号并重发。 | rejected=建议注销账户。
P04 | 问题=地址如何修改？ | chosen=在发货前更新地址。 | rejected=永久不可修改。
P05 | 问题=银行卡被盗刷？ | chosen=冻结卡并保留证据。 | rejected=继续正常使用。
P06 | 问题=发票抬头写错？ | chosen=按规则更正抬头。 | rejected=删除全部订单。


## Baseline / 基线

先运行最朴素、但同样使用这些输入和同一指标的对照，避免只看一个核心算法数字。


In [2]:
rollouts = [{'prompt': '退款是否可审批？', 'answer': '核验订单后可退款。', 'old': -5.0, 'new': -4.4, 'length': 5, 'reward': 1.0}, {'prompt': '退款是否可审批？', 'answer': '先补充订单号。', 'old': -3.0, 'new': -3.1, 'length': 3, 'reward': 0.3}, {'prompt': '退款是否可审批？', 'answer': '直接退款并致歉。', 'old': -7.0, 'new': -6.0, 'length': 7, 'reward': 0.8}, {'prompt': '盗刷如何处理？', 'answer': '先冻结账户再核验。', 'old': -6.0, 'new': -5.3, 'length': 6, 'reward': 1.0}, {'prompt': '盗刷如何处理？', 'answer': '建议稍后再试。', 'old': -3.0, 'new': -3.2, 'length': 3, 'reward': 0.0}, {'prompt': '盗刷如何处理？', 'answer': '联系银行并保存证据。', 'old': -5.0, 'new': -4.6, 'length': 5, 'reward': 0.7}]  # 构造两组可读 prompt 的完整回复和 rollout 统计。
raw_ratios = [math.exp(row['new'] - row['old']) for row in rollouts]  # 用未归一化序列 logprob 差计算 ratio。
baseline_metric = max(raw_ratios)  # 记录最长回答可能被放大的最大 ratio。
print(f'未归一化 sequence ratios={ [round(value, 3) for value in raw_ratios] }，最大={baseline_metric:.3f}')  # 展示长度相关的放大。


未归一化 sequence ratios=[1.822, 0.905, 2.718, 2.014, 0.819, 1.492]，最大=2.718


## 手写核心实现与中间量

核心实现保留 state、ratio、优势、mask 或概率分母等中间量，不用 Trainer 或现成 Agent/Attention 框架遮蔽机制。


In [3]:
normalized_ratios = [math.exp((row['new'] - row['old']) / row['length']) for row in rollouts]  # 按回答长度平均 logprob 差得到序列级 ratio。
group_advantages = []  # 保存各 prompt 内的相对优势。
for prompt in sorted(set(row['prompt'] for row in rollouts)):  # 分别处理每个 prompt 的候选组。
    rewards = [row['reward'] for row in rollouts if row['prompt'] == prompt]  # 收集同 prompt 的 reward。
    mean_reward = sum(rewards) / len(rewards)  # 计算组均值作为相对基线。
    group_advantages.extend([reward - mean_reward for reward in rewards])  # 为组中每条回复计算优势。
clipped_terms = [min(ratio * advantage, max(0.8, min(1.2, ratio)) * advantage) for ratio, advantage in zip(normalized_ratios, group_advantages)]  # 手写 PPO 风格序列 surrogate。
core_metric = max(normalized_ratios)  # 保存归一化后的最大 ratio。
print(f'长度归一 sequence ratios={ [round(value, 3) for value in normalized_ratios] }，优势={ [round(value, 3) for value in group_advantages] }')  # 输出 group 相对信号。
print(f'clipped surrogate={ [round(value, 3) for value in clipped_terms] }，最大 ratio={core_metric:.3f}')  # 输出裁剪中间量。


长度归一 sequence ratios=[1.127, 0.967, 1.154, 1.124, 0.936, 1.083]，优势=[0.433, -0.567, 0.133, 0.3, -0.4, 0.1]
clipped surrogate=[0.489, -0.548, 0.154, 0.337, -0.374, 0.108]，最大 ratio=1.154


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立基线与核心的同口径结果表。
for name, metric in comparison_rows:  # 逐行输出结果表。
    print(f'{name:<8} | 指标={metric:.6f}')  # 显示可读数值对照。


Baseline | 指标=2.718282
核心机制     | 指标=1.153565


## 结果解读

这里只能得出本受控样本上的机制结论。生产应冻结 rollout 版本、保存 old logprob/长度/reward，并用 ratio/clip/KL 分位数做报警；不同实现的 length normalization 不能混比。 生产决策必须进一步看验证集、线上安全指标、算力和版本可追溯性。

## 失败案例

下方先让关键条件真实失效，再展示修复如何改变可观测指标。


In [5]:
long_row = rollouts[2]  # 选择同一 prompt 下长度为七的完整回复。
failure_metric = math.exp(long_row['new'] - long_row['old'])  # 计算未归一化的长回答 ratio。
fix_metric = math.exp((long_row['new'] - long_row['old']) / long_row['length'])  # 计算长度归一后的 ratio。
print(f'失败：长回答未归一 ratio={failure_metric:.3f}；修复：序列平均 logprob ratio={fix_metric:.3f}')  # 展示长度偏置来源。


失败：长回答未归一 ratio=2.718；修复：序列平均 logprob ratio=1.154


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产应冻结 rollout 版本、保存 old logprob/长度/reward，并用 ratio/clip/KL 分位数做报警；不同实现的 length normalization 不能混比。

**常见坑：** 用 sequence sum 却不说明长度口径，或将不同 prompt 的奖励直接混在一组里计算相对优势。

**延伸追问：** 序列 ratio 与 token ratio 在长 CoT 上为何不同？如何保证多轮更新时 old policy 不被错误覆盖？

## 生产差距

实验运行于 CPU/FP32，只有 6 条离线样本，省略了真实 rollout、分布式同步、混合精度、内容安全、数据治理、checkpoint 和监控。上线版本应以受审计的状态、指标和回滚流程替代这些教学变量。


In [6]:
assert len(normalized_ratios) == 6  # 验证六条完整回复都有序列 ratio。
assert core_metric < baseline_metric  # 验证长度归一降低了本组的 ratio 极值。
assert failure_metric > fix_metric  # 验证同一长回复的未归一 ratio 更易放大。
assert len(group_advantages) == 6  # 验证每条回复均在同 prompt 组内计算优势。
